# Reclaimed capacity and queueing

Scheduler queueing delays, and overlap between reclamation decisions and waiting requests whose CPU, memory and GPU requests fit the reclaimed allocation. `additional_analyses/reuse_overlap_online.py` reuses these cells for the online decisions reported in Section 4.5.

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
from cache_paths import require_frozen_cache
ONLINE_CACHE = require_frozen_cache('online_ir2', ['container_events.csv', 'validation.json'])

from cohort_contract import current_contract
LATEST_CONTRACT=current_contract()


## 1. Inputs and checks

In [ ]:
# Inputs and outputs stay inside the repository.
from pathlib import Path
from IPython.display import display as _cleanup_display, HTML as _CleanupHTML

RESULTS = RESULTS_REVISION
RESULTS.mkdir(parents=True, exist_ok=True)

def display(*objects, **kwargs):
    """Display-only terminology; dataframe values and calculation keys are unchanged."""
    for obj in objects:
        html = obj.data if isinstance(obj, _CleanupHTML) else (
            obj._repr_html_() if hasattr(obj, "_repr_html_") else None
        )
        if isinstance(html, str):
            html = html.replace("GPU-Enabled", "Integer-GPU")
            _cleanup_display(_CleanupHTML(html), **kwargs)
        else:
            _cleanup_display(obj, **kwargs)

# ============================================================
# 1. Scheduler log 로드 및 queueing 입력 검증
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display as _cleanup_display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

def print_result_header(result_name, description, criterion):
    print("\n" + "=" * 78)
    print(f"[RESULT] {result_name}")
    print(f"설명: {description}")
    print(f"판정 기준: {criterion}")
    print("=" * 78)

def display_bilingual_table(
    frame: pd.DataFrame,
    labels: dict,
    index_labels=None,
    decimals: int = 2,
):
    shown = frame.rename(
        columns={
            key: f"{en}<br>({ko})"
            for key, (en, ko) in labels.items()
        }
    )

    if index_labels is not None:
        axes = [index_labels] if isinstance(index_labels, tuple) else index_labels
        shown = shown.rename_axis(
            index=[
                f"{en}<br>({ko})"
                for en, ko in axes
            ] if len(axes) > 1 else f"{axes[0][0]}<br>({axes[0][1]})"
        )

    html = shown.to_html(
        escape=True,
        float_format=lambda v: f"{v:,.{decimals}f}",
    ).replace("&lt;br&gt;", "<br>")

    display(HTML(html))


DATA_ROOT = Path(str(PROCESSED_DATA))
RAW_ROOT = RAW_USAGE
PROCESSED = DATA_ROOT / "UsageLog20230401_20251130_20260301_20260831.csv"
RAW_GLOB = "*/hub-usage-*.csv"

TRAIN_START = pd.Timestamp("2023-03-01")
TRAIN_END = pd.Timestamp("2025-12-01")
TEST_START = pd.Timestamp("2026-03-01")
TEST_END = pd.Timestamp("2026-09-01")

RAW_COLUMNS = [
    "container_ids",
    "created_at",
    "scheduled_at",
    "run_at",
    "terminated_at",
    "gpu_allocated",
    "agent",
]

paths = sorted(RAW_ROOT.glob(RAW_GLOB))
if not paths:
    raise FileNotFoundError(f"Raw UsageLog를 찾을 수 없습니다: {RAW_ROOT}")

raw = pd.concat(
    (
        pd.read_csv(
            path,
            usecols=RAW_COLUMNS,
            low_memory=False,
        )
        for path in paths
    ),
    ignore_index=True,
)

raw_rows = len(raw)

for col in ["created_at", "scheduled_at", "run_at", "terminated_at"]:
    raw[col] = pd.to_datetime(raw[col], utc=True, errors="coerce")

# 동일 container가 여러 raw 파일에 중복 저장된 경우
# 가장 최신 terminated_at 행을 남긴다.
raw = (
    raw.sort_values("terminated_at", ascending=False)
       .drop_duplicates("container_ids", keep="first")
       .copy()
)

# processed UsageLog와 ID / created_time / GPU allocation을 대조한다.
processed = pd.read_csv(
    PROCESSED,
    usecols=[
        "container_id",
        "created_time",
        "gpu_allocated(EA)",
    ],
    low_memory=False,
)

joined = processed.merge(
    raw,
    left_on="container_id",
    right_on="container_ids",
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not joined["_merge"].eq("both").all():
    raise ValueError("Processed UsageLog와 raw scheduler log의 container ID 연결 실패")

joined = joined.drop(columns="_merge")

# 논문 분석과 동일하게 KST naive datetime으로 변환한다.
for col in ["created_at", "scheduled_at", "run_at", "terminated_at"]:
    joined[col] = (
        joined[col]
        .dt.tz_convert("Asia/Seoul")
        .dt.tz_localize(None)
    )

joined["created_time"] = pd.to_datetime(
    joined["created_time"],
    errors="coerce",
)

if not joined["created_time"].eq(joined["created_at"]).all():
    raise ValueError("Processed created_time과 raw created_at이 일치하지 않습니다.")

if not np.allclose(
    joined["gpu_allocated(EA)"],
    joined["gpu_allocated"],
    equal_nan=False,
):
    raise ValueError("Processed/raw GPU allocation이 일치하지 않습니다.")

joined["queueing_delay_min"] = (
    joined["run_at"] - joined["created_at"]
).dt.total_seconds() / 60

joined["period"] = "Outside"
joined.loc[
    joined["created_at"].ge(TRAIN_START)
    & joined["created_at"].lt(TRAIN_END),
    "period",
] = "Train"
joined.loc[
    joined["created_at"].ge(TEST_START)
    & joined["created_at"].lt(TEST_END),
    "period",
] = "Test"

gpu = joined["gpu_allocated(EA)"].astype(float)
is_integer_gpu = gpu.gt(0) & np.isclose(gpu, np.round(gpu))
is_fractional_gpu = gpu.gt(0) & ~np.isclose(gpu, np.round(gpu))

joined["request_group"] = np.select(
    [
        gpu.eq(0),
        is_integer_gpu,
        is_fractional_gpu,
    ],
    [
        "Non-GPU",
        "Integer-GPU",
        "Fractional-GPU",
    ],
    default="Unknown",
)

analysis = joined.loc[
    joined["period"].isin(["Train", "Test"])
].copy()

invalid_created_run = int(
    (
        analysis["created_at"].notna()
        & analysis["run_at"].notna()
        & analysis["created_at"].gt(analysis["run_at"])
    ).sum()
)

invalid_run_terminated = int(
    (
        analysis["run_at"].notna()
        & analysis["terminated_at"].notna()
        & analysis["run_at"].gt(analysis["terminated_at"])
    ).sum()
)

audit = (
    analysis.groupby(["period", "request_group"], observed=True)
    .agg(
        requests=("container_id", "size"),
        valid_run=("run_at", "count"),
        missing_run=("run_at", lambda s: int(s.isna().sum())),
    )
)

print_result_header(
    "queueing_input_audit",
    "Scheduler raw log와 processed UsageLog를 연결하고 queueing delay 입력을 검증",
    "ID/GPU allocation 일치, created≤run≤terminated, Train/Test run_at 결측 확인",
)

print(f"Raw rows before latest-record dedup: {raw_rows:,}")
print(f"Rows after container dedup: {len(raw):,}")
print(f"Processed/raw matched containers: {len(joined):,}")
print(f"created > run 사례: {invalid_created_run:,}")
print(f"run > terminated 사례: {invalid_run_terminated:,}")

print("\n표: Train/Test scheduler 요청 및 run_at 관측 수")
display(audit)

if invalid_created_run or invalid_run_terminated:
    raise ValueError("Queueing 계산에 사용하는 lifecycle timestamp 역전 사례가 있습니다.")

if analysis["request_group"].eq("Unknown").any():
    raise ValueError("GPU allocation 유형을 분류할 수 없는 행이 있습니다.")

print("Result: PASS")

# Cohort and applied IR2 timeouts (85/119/146/170) from the frozen cohort contract.
sys.path.insert(0, str(PROJECT_ROOT / "04_Analysis/revision"))
from cohort_contract import current_contract
LATEST_CONTRACT = current_contract()


## 2. Queueing delay 분포

In [ ]:
# ============================================================
# 2. Train/Test queueing delay 분포
# ============================================================

GROUP_ORDER = [
    "All workload",
    "GPU-enabled",
    "Integer-GPU",
    "Non-GPU",
    "Fractional-GPU",
]

summary_rows = []

for period in ["Train", "Test"]:
    base = analysis.loc[
        analysis["period"].eq(period)
        & analysis["queueing_delay_min"].notna()
    ].copy()

    groups = {
        "All workload": base,
        "GPU-enabled": base.loc[base["gpu_allocated(EA)"].gt(0)],
        "Integer-GPU": base.loc[base["request_group"].eq("Integer-GPU")],
        "Non-GPU": base.loc[base["request_group"].eq("Non-GPU")],
        "Fractional-GPU": base.loc[base["request_group"].eq("Fractional-GPU")],
    }

    for group_name, group in groups.items():
        values = group["queueing_delay_min"].to_numpy(dtype=float)

        if len(values) == 0:
            continue

        summary_rows.append({
            "period": period,
            "group": group_name,
            "requests": len(values),
            "p50_min": np.percentile(values, 50),
            "p90_min": np.percentile(values, 90),
            "p95_min": np.percentile(values, 95),
            "p99_min": np.percentile(values, 99),
            "max_min": np.max(values),
            "over_10min": int(np.sum(values > 10)),
            "over_60min": int(np.sum(values > 60)),
            "over_360min": int(np.sum(values > 360)),
        })

queue_summary = pd.DataFrame(summary_rows)

for cutoff in [10, 60, 360]:
    queue_summary[f"over_{cutoff}min_pct"] = (
        100
        * queue_summary[f"over_{cutoff}min"]
        / queue_summary["requests"]
    )

queue_summary["group"] = pd.Categorical(
    queue_summary["group"],
    categories=GROUP_ORDER,
    ordered=True,
)

queue_summary = queue_summary.sort_values(
    ["period", "group"]
).reset_index(drop=True)

RESULTS = RESULTS_REVISION
RESULTS.mkdir(exist_ok=True)

queue_summary.to_csv(
    RESULTS / "05_queueing_delay_summary.csv",
    index=False,
)

print_result_header(
    "queueing_delay_distribution",
    "run_at - created_at으로 정의한 observed queueing delay의 Train/Test 분포",
    "모든 유효 delay가 0 이상이며 count·percentile·threshold count가 내부적으로 일치",
)

table_a = queue_summary[[
    "period", "group", "requests",
    "p50_min", "p90_min", "p95_min", "p99_min", "max_min",
]].copy()

print("표 A: Queueing delay 분포")
display_bilingual_table(
    table_a.set_index(["period", "group"]),
    {
        "requests": ("Requests", "요청 수"),
        "p50_min": ("P50, min", "중앙값·분"),
        "p90_min": ("P90, min", "90분위·분"),
        "p95_min": ("P95, min", "95분위·분"),
        "p99_min": ("P99, min", "99분위·분"),
        "max_min": ("Max, min", "최대값·분"),
    },
    index_labels=[
        ("Population", "Train/Test"),
        ("Request group", "요청 집단"),
    ],
)

table_b = queue_summary[[
    "period", "group", "requests",
    "over_10min", "over_10min_pct",
    "over_60min", "over_60min_pct",
    "over_360min", "over_360min_pct",
]].copy()

print("\n표 B: 긴 queueing delay 요청")
display_bilingual_table(
    table_b.set_index(["period", "group"]),
    {
        "requests": ("Requests", "요청 수"),
        "over_10min": (">10 min", "10분 초과 요청"),
        "over_10min_pct": (">10 min, %", "10분 초과 비율·%"),
        "over_60min": (">60 min", "60분 초과 요청"),
        "over_60min_pct": (">60 min, %", "60분 초과 비율·%"),
        "over_360min": (">360 min", "6시간 초과 요청"),
        "over_360min_pct": (">360 min, %", "6시간 초과 비율·%"),
    },
    index_labels=[
        ("Population", "Train/Test"),
        ("Request group", "요청 집단"),
    ],
)

if (
    queue_summary[
        ["p50_min", "p90_min", "p95_min", "p99_min", "max_min"]
    ] < 0
).any().any():
    raise ValueError("음수 queueing delay 통계가 존재합니다.")

print("Result: PASS")


In [ ]:
# ============================================================
# 3. Integer-GPU queueing delay CCDF
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": True,
    "font.size": 10.5,
    "figure.dpi": 130,
})

fig, ax = plt.subplots(figsize=(8.8, 4.8))

for period in ["Train", "Test"]:
    values = np.sort(
        analysis.loc[
            analysis["period"].eq(period)
            & analysis["request_group"].eq("Integer-GPU")
            & analysis["queueing_delay_min"].notna(),
            "queueing_delay_min",
        ].to_numpy(dtype=float)
    )

    positive = values[values > 0]

    if len(positive) == 0:
        continue

    y = (
        100
        * np.arange(len(values), len(values) - len(positive), -1)
        / len(values)
    )

    ax.step(
        positive,
        y,
        where="post",
        linewidth=1.8,
        label=period,
    )

for x in [10, 60, 360]:
    ax.axvline(
        x,
        linestyle="--",
        linewidth=1.0,
        alpha=0.55,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(0.01, 100)
ax.set_xlabel("Queueing delay (minutes, log scale)")
ax.set_ylabel("Requests with queueing delay longer than x (%) — CCDF")
ax.set_title("Integer-GPU queueing delay: Train vs. Test")
ax.grid(alpha=0.18, which="both")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)

fig.tight_layout()

save_path = RESULTS / "05_integer_gpu_queueing_delay_ccdf.png"
fig.savefig(
    str(save_path),
    dpi=180,
    bbox_inches="tight",
)

from IPython.display import Image
display(Image(filename=str(save_path)))
plt.close(fig)

print(f"[RESULT] Figure saved: {save_path}")


## 3. Queueing delay decomposition

In [ ]:
# ============================================================
# 4. Integer-GPU queueing delay decomposition
# ============================================================

decomp = analysis.loc[
    analysis["request_group"].eq("Integer-GPU")
    & analysis["created_at"].notna()
    & analysis["scheduled_at"].notna()
    & analysis["run_at"].notna()
].copy()

decomp["scheduler_wait_min"] = (
    decomp["scheduled_at"] - decomp["created_at"]
).dt.total_seconds() / 60

decomp["launch_delay_min"] = (
    decomp["run_at"] - decomp["scheduled_at"]
).dt.total_seconds() / 60

bad_scheduler_order = int(
    (decomp["scheduler_wait_min"] < 0).sum()
)
bad_launch_order = int(
    (decomp["launch_delay_min"] < 0).sum()
)

if bad_scheduler_order or bad_launch_order:
    raise ValueError(
        f"Timestamp ordering 오류: "
        f"scheduler_wait<0={bad_scheduler_order}, "
        f"launch_delay<0={bad_launch_order}"
    )

decomp_rows = []

for period in ["Train", "Test"]:
    part = decomp.loc[
        decomp["period"].eq(period)
    ].copy()

    for metric in [
        "queueing_delay_min",
        "scheduler_wait_min",
        "launch_delay_min",
    ]:
        values = part[metric].dropna().to_numpy(dtype=float)

        decomp_rows.append({
            "period": period,
            "metric": metric,
            "requests": len(values),
            "p50_min": np.percentile(values, 50),
            "p90_min": np.percentile(values, 90),
            "p95_min": np.percentile(values, 95),
            "p99_min": np.percentile(values, 99),
            "max_min": np.max(values),
        })

decomp_summary = pd.DataFrame(decomp_rows)

decomp_summary.to_csv(
    RESULTS / "05_queueing_delay_decomposition.csv",
    index=False,
)

print_result_header(
    "queueing_delay_decomposition",
    "integer-GPU queueing delay를 scheduler waiting과 launch delay로 분해",
    "created≤scheduled≤run이 유지되고 long tail이 어느 구간에서 발생하는지 확인",
)

show = decomp_summary.copy()
show["metric"] = show["metric"].replace({
    "queueing_delay_min": "Queueing delay",
    "scheduler_wait_min": "Scheduler waiting",
    "launch_delay_min": "Launch delay",
})

display_bilingual_table(
    show.set_index(["period", "metric"])[[
        "requests",
        "p50_min",
        "p90_min",
        "p95_min",
        "p99_min",
        "max_min",
    ]],
    {
        "requests": ("Requests", "요청 수"),
        "p50_min": ("P50, min", "중앙값·분"),
        "p90_min": ("P90, min", "90분위·분"),
        "p95_min": ("P95, min", "95분위·분"),
        "p99_min": ("P99, min", "99분위·분"),
        "max_min": ("Max, min", "최대값·분"),
    },
    index_labels=[
        ("Population", "Train/Test"),
        ("Component", "지연 구성"),
    ],
)

print("Result: PASS")


## 4. Longest-delay records 확인

In [ ]:
# ============================================================
# 5. Integer-GPU longest queueing delay audit
# ============================================================

top_parts = []

for period in ["Train", "Test"]:
    part = (
        analysis.loc[
            analysis["period"].eq(period)
            & analysis["request_group"].eq("Integer-GPU")
            & analysis["queueing_delay_min"].notna(),
            [
                "container_id",
                "created_at",
                "scheduled_at",
                "run_at",
                "terminated_at",
                "gpu_allocated(EA)",
                "agent",
                "queueing_delay_min",
            ],
        ]
        .nlargest(10, "queueing_delay_min")
        .copy()
    )

    part.insert(0, "period", period)
    part["queueing_delay_hour"] = (
        part["queueing_delay_min"] / 60
    )
    top_parts.append(part)

longest = pd.concat(
    top_parts,
    ignore_index=True,
)

longest.to_csv(
    RESULTS / "05_integer_gpu_longest_queueing_records.csv",
    index=False,
)

print_result_header(
    "longest_queueing_delay_audit",
    "Train/Test integer-GPU의 가장 긴 queueing delay 기록을 lifecycle timestamp와 함께 확인",
    "created≤scheduled≤run≤terminated가 유지되는지 확인하고 원인은 추정하지 않음",
)

display(
    longest[[
        "period",
        "created_at",
        "scheduled_at",
        "run_at",
        "terminated_at",
        "gpu_allocated(EA)",
        "queueing_delay_min",
        "queueing_delay_hour",
        "agent",
    ]]
)

ordered = (
    longest["created_at"].le(longest["scheduled_at"])
    & longest["scheduled_at"].le(longest["run_at"])
    & longest["run_at"].le(longest["terminated_at"])
)

print(
    "Top-10 lifecycle ordering:",
    "PASS" if ordered.all() else "CHECK REQUIRED",
)


## Queueing context의 결과와 해석 범위

## 5. Reclaim–Demand Overlap (회수 시점과 대기 수요의 겹침)

In [ ]:
# [RESULT] reclaim_overlap_input_audit
# 왜: 02의 phase event를 정확한 scheduler container 및 CPU/Memory/GPU 원자료에 연결한다.
import hashlib
import json
import heapq
from collections import defaultdict

CACHE_FILE = ONLINE_CACHE / 'container_events.csv'
CACHE_VALIDATION = CACHE_FILE.parent / 'validation.json'
# External producer is still read only for provenance hashing, not executed.
CACHE_PRODUCER = Path(str(REVISION_CODE / 'online_ir2/simulation.py'))
OVERLAP_RAW_COLUMNS = ['container_ids','created_at','scheduled_at','run_at','terminated_at',
                       'cpu_allocated','mem_allocated','gpu_allocated','agent']

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

validation = json.loads(CACHE_VALIDATION.read_text())
assert validation['checks'] == 'PASS'
source_events = pd.read_csv(CACHE_FILE, parse_dates=[
    'created_time','terminated_time','first_observation','last_observation',
    'first_active','last_active','hindsight_pre_time','hindsight_post_time'])
assert len(source_events) == LATEST_CONTRACT['row_groups'] and source_events.filename.is_unique
assert source_events.population.value_counts().to_dict() == LATEST_CONTRACT['population_counts']
assert np.allclose(source_events.hindsight_pre_saved_min + source_events.hindsight_post_saved_min,
                   source_events.hindsight_saved_min, rtol=0, atol=1e-7)
assert set(source_events.gpu_bin.unique()) == {'Non-GPU','GPU-Enabled'}

# processed filename은 .csv가 붙지만 02 cache filename은 확장자를 제거한다.
meta = pd.read_csv(PROCESSED, usecols=['filename','container_id','created_time',
    'cpu_allocated(Core)','mem_allocated(GB)','gpu_allocated(EA)'], low_memory=False)
meta['event_key'] = meta.filename.str.removesuffix('.csv')
assert meta.event_key.is_unique and meta.container_id.is_unique

# 2절과 같은 raw 중복 제거 규칙을 적용한다. 요구량은 raw allocation이다.
raw_overlap = pd.concat((pd.read_csv(path, usecols=OVERLAP_RAW_COLUMNS, low_memory=False)
    for path in paths), ignore_index=True)
raw_rows_overlap = len(raw_overlap)
raw_missing_id_before_dedup = int(raw_overlap.container_ids.isna().sum())
raw_missing_schedule_before_dedup = int(raw_overlap.scheduled_at.isna().sum())
raw_missing_run_before_dedup = int(raw_overlap.run_at.isna().sum())
for column in ['created_at','scheduled_at','run_at','terminated_at']:
    raw_overlap[column] = pd.to_datetime(raw_overlap[column], utc=True, errors='coerce')
raw_overlap = (raw_overlap.sort_values('terminated_at', ascending=False)
    .drop_duplicates('container_ids', keep='first').copy())
for column in ['created_at','scheduled_at','run_at','terminated_at']:
    raw_overlap[column] = raw_overlap[column].dt.tz_convert('Asia/Seoul').dt.tz_localize(None)
for column in ['cpu_allocated','mem_allocated','gpu_allocated']:
    raw_overlap[column] = pd.to_numeric(raw_overlap[column], errors='coerce')

linked = source_events.merge(meta, left_on='filename', right_on='event_key',
    how='left', validate='one_to_one', indicator=True, suffixes=('','_processed'))
cache_processed_matches = int(linked['_merge'].eq('both').sum())
linked = linked.drop(columns='_merge').merge(raw_overlap, left_on='container_id',
    right_on='container_ids', how='left', validate='one_to_one', indicator=True,
    suffixes=('','_raw'))
cache_raw_matches = int(linked['_merge'].eq('both').sum())
linked = linked.drop(columns='_merge')
resource_match = bool(
    np.allclose(linked.cpu_allocated, linked['cpu_allocated(Core)'], rtol=0, atol=1e-9)
    and np.allclose(linked.cpu_allocated, linked.cpu_allocated_raw, rtol=0, atol=1e-9)
    and np.allclose(linked.gpu_allocated, linked['gpu_allocated(EA)'], rtol=0, atol=1e-9)
    and np.allclose(linked.gpu_allocated, linked.gpu_allocated_raw, rtol=0, atol=1e-9)
    and np.allclose(linked['mem_allocated(GB)'], linked.mem_allocated, rtol=0, atol=1e-9)
)
created_match = bool(pd.to_datetime(linked.created_time).eq(
    pd.to_datetime(linked.created_time_processed)).all()
    and pd.to_datetime(linked.created_time).eq(linked.created_at).all())
assert cache_processed_matches == cache_raw_matches == len(source_events)
assert resource_match and created_match
assert linked.container_id.is_unique

# Scheduler-wait만 primary demand: created <= t < scheduled. 이미 placed된 launch 구간은 제외.
resource_valid = (raw_overlap[['cpu_allocated','mem_allocated','gpu_allocated']].notna().all(axis=1)
    & np.isfinite(raw_overlap[['cpu_allocated','mem_allocated','gpu_allocated']]).all(axis=1)
    & raw_overlap[['cpu_allocated','mem_allocated','gpu_allocated']].ge(0).all(axis=1))
ordering_valid = (raw_overlap.created_at.notna() & raw_overlap.scheduled_at.notna()
    & raw_overlap.created_at.le(raw_overlap.scheduled_at)
    & (raw_overlap.run_at.isna() | raw_overlap.scheduled_at.le(raw_overlap.run_at))
    & (raw_overlap.run_at.isna() | raw_overlap.terminated_at.isna()
       | raw_overlap.run_at.le(raw_overlap.terminated_at)))
request_valid = raw_overlap.container_ids.notna() & resource_valid & ordering_valid
waiting_requests = raw_overlap.loc[request_valid & raw_overlap.created_at.lt(raw_overlap.scheduled_at)].copy()
waiting_requests['request_period'] = np.select([
    waiting_requests.created_at.ge(TRAIN_START) & waiting_requests.created_at.lt(TRAIN_END),
    waiting_requests.created_at.ge(TEST_START) & waiting_requests.created_at.lt(TEST_END)],
    ['Train','Test'], default='Outside')
waiting_requests['request_population'] = np.select([
    waiting_requests.gpu_allocated.eq(0),
    waiting_requests.gpu_allocated.gt(0) & np.isclose(waiting_requests.gpu_allocated,
                                                     np.round(waiting_requests.gpu_allocated)),
    waiting_requests.gpu_allocated.gt(0)],
    ['Non-GPU','Integer-GPU','Fractional-GPU'], default='Unknown')

input_audit = {
    'producer_notebook': '02. OnlineIR2Simulation.ipynb',
    'cache_file': str(CACHE_FILE), 'cache_sha256': sha256_file(CACHE_FILE),
    'cache_validation_file': str(CACHE_VALIDATION),
    'producer_engine': str(CACHE_PRODUCER), 'producer_engine_sha256': sha256_file(CACHE_PRODUCER),
    'processed_usagelog': str(PROCESSED), 'raw_scheduler_files': [str(p) for p in paths],
    'source_event_containers': len(source_events), 'cache_processed_id_matches':cache_processed_matches,
    'cache_raw_id_matches': cache_raw_matches, 'created_timestamp_matches':created_match,
    'cpu_gpu_memory_matches': resource_match,
    'raw_rows':raw_rows_overlap, 'raw_deduplicated_rows':len(raw_overlap),
    'raw_missing_container_id_before_dedup':raw_missing_id_before_dedup,
    'raw_missing_scheduled_at_before_dedup':raw_missing_schedule_before_dedup,
    'raw_missing_run_at_before_dedup':raw_missing_run_before_dedup,
    'raw_missing_container_id':int(raw_overlap.container_ids.isna().sum()),
    'raw_missing_scheduled_at':int(raw_overlap.scheduled_at.isna().sum()),
    'raw_missing_run_at':int(raw_overlap.run_at.isna().sum()),
    'raw_invalid_resources':int((~resource_valid).sum()),
    'raw_invalid_lifecycle_order':int((~ordering_valid).sum()),
    'valid_positive_scheduler_wait_requests':len(waiting_requests),
    'timestamp_timezone':'KST naive; raw UTC converted to Asia/Seoul',
    'request_window':'created_at <= reclaim_time < scheduled_at',
    'queueing_decomposition':'run-created = (scheduled-created) + (run-scheduled)',
}
print_result_header('reclaim_overlap_input_audit',
    '02 frozen hindsight cache, processed UsageLog, raw scheduler log의 ID·시간·자원 연결',
    '15,125/15,125 ID 및 CPU/GPU/Memory 일치; 유효 scheduler-wait 요청만 사용')
display(pd.DataFrame([{'Cache events':len(source_events),
    'ID matched':cache_raw_matches,'Raw rows':raw_rows_overlap,
    'Raw missing ID (before dedup)':raw_missing_id_before_dedup,
    'Raw missing scheduled (before dedup)':raw_missing_schedule_before_dedup,
    'Raw missing run (before dedup)':raw_missing_run_before_dedup,
    'Valid waiting requests':len(waiting_requests),
    'Invalid lifecycle rows':int((~ordering_valid).sum()),
    'Invalid allocation rows':int((~resource_valid).sum())}]))
print('Result: PASS (invalid raw rows excluded and counted)')


## 6. Virtual reclaim events (가상 회수 사건)

In [ ]:
# [RESULT] virtual_reclaim_event_summary
phase_frames = []
for phase in ['Pre','Post']:
    low = phase.lower()
    part = linked.loc[linked[f'hindsight_{low}_saved_min'].gt(0)].copy()
    part['phase'] = phase
    part['reclaim_time'] = part[f'hindsight_{low}_time']
    part['potentially_reclaimable_min'] = part[f'hindsight_{low}_saved_min']
    phase_frames.append(part)
virtual = pd.concat(phase_frames, ignore_index=True)
virtual['population_group'] = virtual.gpu_bin.replace({'GPU-Enabled':'Integer-GPU'})
virtual['event_id'] = virtual.filename + '|' + virtual.phase
virtual['reclaim_cpu_core'] = virtual.cpu_allocated
virtual['reclaim_mem_gb'] = virtual['mem_allocated(GB)']
virtual['reclaim_gpu_ea'] = virtual.gpu_allocated
virtual['reclaim_agent'] = virtual.agent
virtual['positive_duration'] = virtual.potentially_reclaimable_min.gt(0)
virtual['lifecycle_valid'] = (virtual.reclaim_time.notna()
    & virtual.reclaim_time.ge(virtual.created_time)
    & virtual.reclaim_time.le(virtual.terminated_time))
virtual['resource_valid'] = (virtual[['reclaim_cpu_core','reclaim_mem_gb','reclaim_gpu_ea']]
    .notna().all(axis=1) & np.isfinite(virtual[['reclaim_cpu_core','reclaim_mem_gb','reclaim_gpu_ea']]).all(axis=1)
    & virtual[['reclaim_cpu_core','reclaim_mem_gb','reclaim_gpu_ea']].ge(0).all(axis=1))
virtual['scheduler_log_covered'] = (
    (virtual.population.eq('Train') & virtual.reclaim_time.ge('2023-04-01')
     & virtual.reclaim_time.lt('2025-12-01'))
    | (virtual.population.eq('Test') & virtual.reclaim_time.ge('2026-03-01')
       & virtual.reclaim_time.lt('2026-09-01')))
virtual['evaluable'] = (virtual.positive_duration & virtual.lifecycle_valid
    & virtual.resource_valid & virtual.scheduler_log_covered)
virtual['check_reason'] = np.select([
    ~virtual.lifecycle_valid, ~virtual.resource_valid, ~virtual.scheduler_log_covered],
    ['event_outside_container_lifecycle','invalid_reclaim_allocation','outside_raw_log_window'], default='')
assert virtual.event_id.is_unique
assert len(virtual) == int(source_events.hindsight_pre_saved_min.gt(0).sum()
                           + source_events.hindsight_post_saved_min.gt(0).sum())
pre_later_active = (source_events.loc[source_events.hindsight_pre_saved_min.gt(0)
    & source_events.first_active.notna()].groupby('population').size().to_dict())
assert pre_later_active == LATEST_CONTRACT['pre_positive_later_active']
assert virtual.population_group.isin(['Non-GPU','Integer-GPU']).all()
assert virtual.positive_duration.all()
assert np.array_equal(virtual.loc[virtual.phase.eq('Pre'),'reclaim_time'].astype('int64'),
    (virtual.loc[virtual.phase.eq('Pre'),'first_observation']
     + pd.to_timedelta(np.where(virtual.loc[virtual.phase.eq('Pre'),'population_group'].eq('Non-GPU'),
                                   85,119),unit='m')).astype('int64'))  # GPU Pre 119
assert np.array_equal(virtual.loc[virtual.phase.eq('Post'),'reclaim_time'].astype('int64'),
    (virtual.loc[virtual.phase.eq('Post'),'last_active']
     + pd.to_timedelta(np.where(virtual.loc[virtual.phase.eq('Post'),'population_group'].eq('Non-GPU'),
                                   146,170),unit='m')).astype('int64'))
virtual_summary = (virtual.groupby(['population','phase','population_group'], observed=True)
    .agg(reclaim_events=('event_id','size'),unique_containers=('filename','nunique'),
         evaluable_events=('evaluable','sum'))
    .rename_axis(index=['Period','Phase','Population']).reset_index())
virtual_summary['check_events'] = virtual_summary.reclaim_events - virtual_summary.evaluable_events
virtual_summary.to_csv(RESULTS/'05_virtual_reclaim_event_summary.csv',index=False)
input_audit.update({'virtual_reclaim_events':len(virtual),
    'unique_reclaim_containers':int(virtual.filename.nunique()),
    'pre_positive_later_active_by_period':pre_later_active,
    'both_phase_containers':int(((source_events.hindsight_pre_saved_min>0)
        & (source_events.hindsight_post_saved_min>0)).sum()),
    'events_outside_lifecycle':int((~virtual.lifecycle_valid).sum()),
    'events_outside_raw_log_window':int((~virtual.scheduler_log_covered).sum()),
    'evaluable_events':int(virtual.evaluable.sum()),
    'check_reason_counts':virtual.loc[~virtual.evaluable,'check_reason'].value_counts().to_dict()})
print_result_header('virtual_reclaim_event_summary',
    'Train/Test × Pre/Post × Non-GPU/Integer-GPU의 phase event 및 unique container',
    '양의 02 hindsight duration만 사용; frozen 시각 재현; 비평가 사건은 CHECK 분리')
display(virtual_summary)
print('Overall:',len(virtual),'events /',virtual.filename.nunique(),
      'unique containers /',int(virtual.evaluable.sum()),'evaluable')
print('Result:', 'PASS' if virtual.evaluable.all() else 'CHECK (outside lifecycle/log window)')


## 7. Contemporaneous scheduler waiting (동시 scheduler 대기)

In [ ]:
# [RESULT] contemporaneous_waiting_demand
# 왜: 시간순 sweep으로 4천여 회수 사건에 당시 scheduler-wait 구간만 연결한다.
req = waiting_requests.sort_values('created_at').reset_index(drop=True)
request_ids = req.container_ids.to_numpy(dtype=object)
request_agents = req.agent.to_numpy(dtype=object)
request_cpu = req.cpu_allocated.to_numpy(dtype=float)
request_mem = req.mem_allocated.to_numpy(dtype=float)
request_gpu = req.gpu_allocated.to_numpy(dtype=float)
request_period = req.request_period.to_numpy(dtype=object)
request_population = req.request_population.to_numpy(dtype=object)
request_has_run = req.run_at.notna().to_numpy()
request_start = req.created_at.to_numpy(dtype='datetime64[ns]').astype('int64')
request_end = req.scheduled_at.to_numpy(dtype='datetime64[ns]').astype('int64')

detail_rows = []
unique_waiting = defaultdict(set)
active = set()
end_heap = []
cursor = 0
for event in virtual.loc[virtual.evaluable].sort_values('reclaim_time').itertuples(index=False):
    t = event.reclaim_time.value
    while cursor < len(req) and request_start[cursor] <= t:
        if request_end[cursor] > t:
            active.add(cursor)
            heapq.heappush(end_heap,(request_end[cursor],cursor))
        cursor += 1
    while end_heap and end_heap[0][0] <= t:
        _, expired = heapq.heappop(end_heap)
        active.discard(expired)
    candidates = np.fromiter(active,dtype=np.int64)
    candidates = candidates[request_ids[candidates] != event.container_id]
    key = (event.population,event.phase,event.population_group)
    unique_waiting[key].update(request_ids[candidates])
    # 원자원 vector를 동시에 충족해야 한 event가 한 waiting request를 온전히 수용 가능하다.
    compatible = candidates[(request_cpu[candidates] <= event.reclaim_cpu_core + 1e-9)
        & (request_mem[candidates] <= event.reclaim_mem_gb + 1e-9)
        & (request_gpu[candidates] <= event.reclaim_gpu_ea + 1e-9)]
    # agent는 request가 *나중에 실행된* node일 뿐 당시의 hard constraint가 아니다.
    same_agent = compatible[(request_agents[compatible] == event.reclaim_agent)
                            & request_has_run[compatible]]
    detail_rows.append({'event_id':event.event_id,'waiting_requests':len(candidates),
        'compatible_requests':len(compatible),'same_agent_requests':len(same_agent),
        'fractional_gpu_waiting_requests':int(np.count_nonzero(
            request_population[candidates]=='Fractional-GPU')),
        'cross_period_waiting_requests':int(np.count_nonzero(request_period[candidates]!=event.population))})

detail = virtual[['event_id','filename','container_id','population','phase','population_group',
    'reclaim_time','reclaim_cpu_core','reclaim_mem_gb','reclaim_gpu_ea','reclaim_agent',
    'potentially_reclaimable_min','evaluable','check_reason']].rename(columns={
        'population':'Period','phase':'Phase','population_group':'Population'})
detail = detail.merge(pd.DataFrame(detail_rows),on='event_id',how='left',validate='one_to_one')
for column in ['waiting_requests','compatible_requests','same_agent_requests',
               'fractional_gpu_waiting_requests','cross_period_waiting_requests']:
    detail[column] = detail[column].fillna(0).astype(int)
detail['L1_any_waiting'] = detail.waiting_requests.gt(0)
detail['L2_resource_compatible'] = detail.compatible_requests.gt(0)
detail['L3_same_agent_diagnostic'] = detail.same_agent_requests.gt(0)
detail['funnel_level'] = np.select([
    ~detail.evaluable,detail.L3_same_agent_diagnostic,detail.L2_resource_compatible,
    detail.L1_any_waiting],['CHECK','L3','L2','L1'],default='L0')
assert (detail.same_agent_requests <= detail.compatible_requests).all()
assert (detail.compatible_requests <= detail.waiting_requests).all()
assert len(detail)==len(virtual) and detail.event_id.is_unique

def summarize_demand(part, key):
    observed = part.loc[part.evaluable]
    values = observed.waiting_requests.to_numpy(dtype=float)
    return {'Period':key[0],'Phase':key[1],'Population':key[2],
        'Reclaim events':len(part),'Evaluable events':len(observed),
        'Any waiting demand':int(observed.L1_any_waiting.sum()),
        'Temporal overlap %':100*observed.L1_any_waiting.mean(),
        'Waiting P50':np.percentile(values,50),'Waiting P90':np.percentile(values,90),
        'Waiting P95':np.percentile(values,95),'Waiting max':values.max(),
        'Unique waiting requests':len(unique_waiting[key]),
        'Fractional-GPU waiting pairs (diagnostic)':int(observed.fractional_gpu_waiting_requests.sum()),
        'Cross-period waiting pairs (diagnostic)':int(observed.cross_period_waiting_requests.sum())}
waiting_summary = pd.DataFrame([summarize_demand(part,key) for key,part in
    detail.groupby(['Period','Phase','Population'],observed=True)])
waiting_summary.to_csv(RESULTS/'05_contemporaneous_waiting_demand.csv',index=False)
print_result_header('contemporaneous_waiting_demand',
    'created≤reclaim<scheduled; 자기 ID 제외; 전체 raw 요청 모집단',
    'L1은 evaluable event 중 하나 이상 대기 요청이 있는 사건; P50/P90/P95/max 포함')
display(waiting_summary)
print('Result: PASS (cross-period and fractional requests retained as diagnostics)')


## 8. Full-vector compatibility와 same-agent diagnostic

In [ ]:
# [RESULT] reclaim_resource_compatibility / same_agent_placement_diagnostic
result_rows = []
for key, part in detail.groupby(['Period','Phase','Population'],sort=True,observed=True):
    valid = part.loc[part.evaluable]
    l1 = int(valid.L1_any_waiting.sum())
    l2 = int(valid.L2_resource_compatible.sum())
    l3 = int(valid.L3_same_agent_diagnostic.sum())
    result_rows.append(dict(zip(['Period','Phase','Population'],key),
        **{'Reclaim events':len(part),'Evaluable events':len(valid),'CHECK events':len(part)-len(valid),
           'L0 no waiting':len(valid)-l1,'Any waiting demand':l1,
           'Temporal overlap %':100*l1/len(valid),'Resource-compatible events':l2,
           'Resource-compatible %':100*l2/l1 if l1 else np.nan,
           'Same-agent diagnostic events':l3,
           'Same-agent %':100*l3/l2 if l2 else np.nan}))
overlap_summary = pd.DataFrame(result_rows).sort_values(
    ['Period','Phase','Population']).reset_index(drop=True)
compatibility_summary = overlap_summary[['Period','Phase','Population',
    'Any waiting demand','Resource-compatible events','Resource-compatible %']].copy()
same_agent_summary = overlap_summary[['Period','Phase','Population',
    'Resource-compatible events','Same-agent diagnostic events','Same-agent %']].copy()
compatibility_summary.to_csv(RESULTS/'05_reclaim_resource_compatibility.csv',index=False)
same_agent_summary.to_csv(RESULTS/'05_same_agent_placement_diagnostic.csv',index=False)
overlap_summary.to_csv(RESULTS/'05_reclaim_demand_overlap_summary.csv',index=False)
detail.to_csv(RESULTS/'05_reclaim_demand_overlap_event_detail.csv',index=False)
input_audit['funnel_counts'] = detail.funnel_level.value_counts().to_dict()
(RESULTS/'05_reclaim_overlap_input_audit.json').write_text(
    json.dumps(input_audit,indent=2,ensure_ascii=False,default=int)+'\n')
print_result_header('reclaim_resource_compatibility',
    'L1 중 CPU/Memory/GPU full-vector에 들어맞는 request ≥1의 event 수',
    'L2≤L1; 분모는 temporal overlap events; request 또는 event 자원을 합산하지 않음')
display(compatibility_summary)
print('Result: PASS')
print_result_header('same_agent_placement_diagnostic',
    'L2 중 eventual execution agent가 reclaiming agent와 동일한 request ≥1의 event 수',
    'L3≤L2; 분모는 resource-compatible events; placement proof로 해석하지 않음')
display(same_agent_summary)
print('Result: PASS')
print_result_header('reclaim_demand_overlap_summary',
    'Train/Test × Pre/Post × Non-GPU/Integer-GPU의 L0→L1→L2→L3 funnel',
    '8개 stratum 완비, 단계별 event denominator 명시, CHECK event 분리')
display(overlap_summary)
assert len(overlap_summary)==8
assert (overlap_summary['L0 no waiting']+overlap_summary['Any waiting demand']
    == overlap_summary['Evaluable events']).all()
assert (overlap_summary['Same-agent diagnostic events']
    <= overlap_summary['Resource-compatible events']).all()
assert (overlap_summary['Resource-compatible events']
    <= overlap_summary['Any waiting demand']).all()
# 독립 검산: sorted endpoint 공식으로 모든 L1 event count를 다시 검증한다.
checked = detail.loc[detail.evaluable]
times = checked.reclaim_time.to_numpy(dtype='datetime64[ns]').astype('int64')
request_index = {identifier:i for i,identifier in enumerate(request_ids)}
own_active = np.array([int((i:=request_index.get(identifier)) is not None
    and request_start[i] <= t < request_end[i])
    for identifier,t in zip(checked.container_id,times)])
independent_counts = (np.searchsorted(request_start,times,side='right')
    - np.searchsorted(np.sort(request_end),times,side='right') - own_active)
assert np.array_equal(independent_counts,checked.waiting_requests.to_numpy())
# 각 stratum에서 5건씩 독립 boolean mask로 full-vector 및 same-agent 수를 대조한다.
spot = pd.concat([g.sample(min(5,len(g)),random_state=20260928)
    for _,g in checked.groupby(['Period','Phase','Population'],observed=True)])
for row in spot.itertuples(index=False):
    t = row.reclaim_time.value
    active_mask = (request_start<=t) & (t<request_end) & (request_ids!=row.container_id)
    vector_mask = (request_cpu<=row.reclaim_cpu_core+1e-9)
    vector_mask &= (request_mem<=row.reclaim_mem_gb+1e-9)
    vector_mask &= (request_gpu<=row.reclaim_gpu_ea+1e-9)
    assert int(active_mask.sum()) == row.waiting_requests
    assert int((active_mask & vector_mask).sum()) == row.compatible_requests
    assert int((active_mask & vector_mask & request_has_run
        & (request_agents==row.reclaim_agent)).sum()) == row.same_agent_requests
print('Independent verification: all waiting counts + 40 full-vector spot checks PASS')
print('Result:', 'PASS' if detail.evaluable.all() else 'CHECK (non-evaluable virtual events separated)')


## 조건 충족 사건의 잠재 회수 자원시간

In [ ]:
# [RESULT] reclaim_demand_overlap_resource_time_summary
# 입력: 앞 절 overlap 분석의 detail, overlap_summary
# 자원시간 비중: 같은 기간·단계·집단끼리 비교. Total은 기간별 전체끼리 비교.

resource_specs = [
    ("reclaim_cpu_core", "potential_cpu_core_min",
     "total_hindsight_cpu_core_min", "cpu_share_of_total_hindsight_pct"),
    ("reclaim_mem_gb", "potential_memory_gb_min",
     "total_hindsight_memory_gb_min", "memory_share_of_total_hindsight_pct"),
    ("reclaim_gpu_ea", "potential_gpu_min",
     "total_hindsight_gpu_min", "gpu_share_of_total_hindsight_pct"),
]

assert detail["event_id"].is_unique
required = ["potentially_reclaimable_min"] + [s[0] for s in resource_specs]
assert detail[required].notna().all().all()
assert np.isfinite(detail[required].to_numpy(dtype=float)).all()
assert detail["potentially_reclaimable_min"].gt(0).all()
assert detail[required[1:]].ge(0).all().all()


def summarize_resource_time(all_events, period, phase, population):
    events = all_events.loc[all_events["evaluable"]]
    compatible = events.loc[events["L2_resource_compatible"]]
    a = len(events)
    b = int(events["L1_any_waiting"].sum())
    d = len(compatible)
    assert a > 0 and 0 <= d <= b <= a

    row = {
        "period": period, "phase": phase, "population": population,
        "analysis_reclaim_events_a": a,
        "events_with_waiting_b": b,
        "waiting_event_pct_c": 100 * b / a,
        "events_with_compatible_request_d": d,
        "compatible_event_pct_e": 100 * d / a,
    }
    for allocation, numerator, denominator, percentage in resource_specs:
        # 분자: 평가 가능하고 조건을 충족한 사건의 자원시간
        row[numerator] = float((
            compatible[allocation] * compatible["potentially_reclaimable_min"]
        ).sum())
        # 분모: 같은 기간·단계·집단의 전체 Hindsight 자원시간
        # hindsight 회수량과 비교하므로 평가 제외 사건도 분모에는 포함
        row[denominator] = float((
            all_events[allocation] * all_events["potentially_reclaimable_min"]
        ).sum())
        # Non-GPU의 GPU 분모는 0: 비율을 계산하지 않음
        row[percentage] = (
            100 * row[numerator] / row[denominator]
            if row[denominator] > 0 else np.nan
        )
    return row


resource_time_rows = []
for period in ["Train", "Test"]:
    period_events = detail.loc[detail["Period"].eq(period)]
    for phase in ["Pre", "Post"]:
        for population in ["Non-GPU", "Integer-GPU"]:
            all_events = period_events.loc[
                period_events["Phase"].eq(phase)
                & period_events["Population"].eq(population)
            ]
            row = summarize_resource_time(all_events, period, phase, population)

            existing = overlap_summary.loc[
                overlap_summary["Period"].eq(period)
                & overlap_summary["Phase"].eq(phase)
                & overlap_summary["Population"].eq(population)
            ]
            assert len(existing) == 1
            existing = existing.iloc[0]
            assert row["analysis_reclaim_events_a"] == existing["Evaluable events"]
            assert row["events_with_waiting_b"] == existing["Any waiting demand"]
            assert row["events_with_compatible_request_d"] == existing["Resource-compatible events"]
            resource_time_rows.append(row)

    # Total: 같은 기간의 두 집단·Pre/Post 전체를 합산
    resource_time_rows.append(
        summarize_resource_time(period_events, period, "Total", "All")
    )

resource_time_summary = pd.DataFrame(resource_time_rows)
assert len(resource_time_summary) == 10

# Total의 건수·자원시간은 합산하고, 비율은 합산된 분자/분모로 재계산
sum_columns = [
    "analysis_reclaim_events_a", "events_with_waiting_b",
    "events_with_compatible_request_d",
] + [column for spec in resource_specs for column in spec[1:3]]
for period in ["Train", "Test"]:
    part = resource_time_summary.loc[resource_time_summary["period"].eq(period)]
    total = part.loc[part["phase"].eq("Total")].iloc[0]
    body = part.loc[part["phase"].ne("Total")]
    for column in sum_columns:
        assert np.isclose(body[column].sum(), total[column])

save_path = RESULTS / "05_reclaim_demand_overlap_resource_time_summary.csv"
resource_time_summary.to_csv(
    save_path, index=False, encoding="utf-8-sig", float_format="%.10f"
)

print_result_header(
    "reclaim_demand_overlap_resource_time_summary",
    "조건 충족 사건의 잠재 회수 자원시간과 같은 단계·집단 회수량 대비 비중",
    "기존 사건 수 일치, Train/Test Total 검산, 0인 자원시간 분모는 비율 제외",
)

labels = {
    "analysis_reclaim_events_a": ("Analysis events (a)", "분석 대상 회수 사건 수"),
    "events_with_waiting_b": ("Waiting events (b)", "대기 요청 존재 사건 수"),
    "waiting_event_pct_c": ("c = b/a × 100, %", "대기 요청 존재 비율"),
    "events_with_compatible_request_d": ("Compatible events (d)", "조건 충족 사건 수"),
    "compatible_event_pct_e": ("e = d/a × 100, %", "조건 충족 사건 비율"),
    "potential_cpu_core_min": ("CPU, core·min", "잠재 회수 CPU 자원시간"),
    "cpu_share_of_total_hindsight_pct": ("CPU share, %", "해당 CPU 잠재 회수량 대비"),
    "potential_memory_gb_min": ("Memory, GB·min", "잠재 회수 Memory 자원시간"),
    "memory_share_of_total_hindsight_pct": ("Memory share, %", "해당 Memory 잠재 회수량 대비"),
    "potential_gpu_min": ("GPU, GPU·min", "잠재 회수 GPU 자원시간"),
    "gpu_share_of_total_hindsight_pct": ("GPU share, %", "해당 GPU 잠재 회수량 대비"),
}
shown = resource_time_summary.set_index(["period", "phase", "population"])
shown = shown[list(labels)].copy()
shown = shown.rename(index={"All": "전체"}, level="population")
shown["gpu_share_of_total_hindsight_pct"] = (
    shown["gpu_share_of_total_hindsight_pct"].astype(object)
    .where(shown["gpu_share_of_total_hindsight_pct"].notna(), "—")
)
display_bilingual_table(
    shown, labels, decimals=3,
    index_labels=[("Period", "기간"), ("Phase", "단계"), ("Population", "구분")],
)
print("자원시간은 잠재 회수량이며 실제 재사용량이 아닙니다.")
print(f"Saved: {save_path}")
print("Result: PASS")


## 9. 결과 해석과 claim boundary (주장 범위)